In [1]:
!pip install faiss-cpu langchain langchain-community langchain-classic langchain-core langchain-huggingface pypdf sentence-transformers transformers==4.52.4 torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 80.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 61.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingf

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import torch
import re

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

def generate_text(prompt, max_length=1000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [6]:
import json

In [15]:
response_schemas = [
    ResponseSchema(
        name="full_name", 
        description="Candidate's full name", 
        type="string"
    ),
    ResponseSchema(
        name="email", 
        description="Candidate's email address", 
        type="string"
    ),
    ResponseSchema(
        name="education",
        description="List of educational qualifications, where each item is an object with 'degree', 'institution', and 'year' (as integer)",
        type="List[dict]",
    ),
    ResponseSchema(
        name="skills",
        description="List of candidate skills as strings",
        type="List[string]",
    ),
    ResponseSchema(
        name="experience",
        description="List of work experiences, where each item is an object with 'role', 'company', and 'years'",
        type="List[dict]",
    ),
]


output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

info_extraction_template = """
You are a smart assistant that extracts information from a user's cv.

Extract all information regarding the name of the user, his email, his education, his skills, and his experience.

Respond ONLY in JSON format as follows:
{format_instructions}

Example Input:
"Mohamed Negm - moahamed123@email.com
Education: B.Sc. Social Science, KFC, 2019
Skills: cooking, selling, cleaning
Experience:
- Worked at MC as a chef assistant (2020-2023)
- Chef at LHC (2023-Present)
"

Now extract from the following input:
"{user_input}"
"""

In [16]:
sample_input = """
John Smith - john.smith@email.com
Education: B.Sc. Computer Science, MIT, 2020
Skills: Python, Machine Learning, Data Analysis
Experience:
- Software Engineer at Google (2020-2023)
- Data Scientist at OpenAI (2023-Present)
"""

prompt = PromptTemplate(
    template=info_extraction_template,
    input_variables=["user_input"],
    partial_variables={"format_instructions": format_instructions},
)

formatted_prompt_str = prompt.format(user_input=sample_input)

response = generate_text(formatted_prompt_str, max_length=1000)[0]

print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



You are a smart assistant that extracts information from a user's cv.

Extract all information regarding the name of the user, his email, his education, his skills, and his experience.

Respond ONLY in JSON format as follows:
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // Candidate's full name
	"email": string  // Candidate's email address
	"education": List[dict]  // List of educational qualifications, where each item is an object with 'degree', 'institution', and 'year' (as integer)
	"skills": List[string]  // List of candidate skills as strings
	"experience": List[dict]  // List of work experiences, where each item is an object with 'role', 'company', and 'years'
}
```

Example Input:
"Mohamed Negm - moahamed123@email.com
Education: B.Sc. Social Science, KFC, 2019
Skills: cooking, selling, cleaning
Experience:
- Worked at MC as a chef assistant (2020-2023)
- 

In [17]:
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"

In [18]:
json_text = extract_json_block(response)
print(json_text)

```json
{
	"full_name": "John Smith",
	"email": "john.smith@email.com",
	"education": [
		{
			"degree": "B.Sc. Computer Science",
			"institution": "MIT",
			"year": 2020
		}
	],
	"skills": [
		"Python",
		"Machine Learning",
		"Data Analysis"
	],
	"experience": [
		{
			"role": "Software Engineer",
			"company": "Google",
			"years": "2020-2023"
		},
		{
			"role": "Data Scientist",
			"company": "OpenAI",
			"years": "2023-Present"
		}
	]
}
```
